In [0]:
dbutils.widgets.text("catalog", "ecom_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
source_path = f"/Volumes/{catalog}/landing/raw/olist/olist_orders_dataset.csv"

df = spark.read.option("header", True).csv(source_path)

display(df)
print(df.count())

In [0]:
from pyspark.sql import functions as F

bronze_df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

bronze_df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.orders")

In [0]:
display(spark.sql(f"""
    SELECT order_status, COUNT(*) AS orders
    FROM {catalog}.bronze.orders
    GROUP BY order_status
    ORDER BY orders DESC
"""))